In [1]:
from pathlib import Path
import muon as mu
import pandas as pd

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
def save_multiomic_count_data_to_csv(h5mu_file, rna_csv_file, atac_csv_file):
    mdata = mu.read(h5mu_file)

    mu.pp.intersect_obs(mdata)

    rna = mdata.mod["rna"]
    rna_gene_by_cell_counts = pd.DataFrame(rna.layers["counts"].todense().T, index=rna.var_names, columns=rna.obs_names)

    atac = mdata.mod["atac"]
    atac_peak_by_cell_counts = pd.DataFrame(atac.layers["counts"].todense().T, index=atac.var_names, columns=atac.obs_names)

    print(f"RNA cells: {rna.n_obs}, RNA genes: {rna.n_vars}")
    print(f"ATAC cells: {atac.n_obs}, ATAC peaks: {atac.n_vars}")

    print(f"Saving RNA data to {rna_csv_file}")
    rna_gene_by_cell_counts.to_csv(rna_csv_file, index=True)
    print(f"Saving ATAC data to {atac_csv_file}")
    atac_peak_by_cell_counts.to_csv(atac_csv_file, index=True)

In [ ]:
experiment_name = f"mouse_liver"

sample_processed_muon_dir = Path(f"/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/processed/{experiment_name}")

h5mu_file = f"/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/processed/{experiment_name}/liver_sample.h5mu"

mdata = mu.read(h5mu_file)

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)


MuData object with n_obs × n_vars = 24457 × 168658
  var:	'gene_ids', 'feature_types', 'genome', 'interval', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
  2 modalities
    rna:	21453 x 19265
      obs:	'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden'
      var:	'gene_ids', 'feature_types', 'genome', 'interval', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
      uns:	'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
      obsm:	'X_pca', 'X_umap'
      varm:	'PCs'
      layers:	'counts'
      obsp:	'connectivities', 'distances'
    atac:	22666 x 149393
      obs:	'n_genes_by_counts', 'total_counts', 'n_counts', 'leiden'
      var:	'gene_ids', 'feature_types', 'genome', 'interval', 'n_cells_by_counts', 'mean_counts'

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [5]:
sample_nums = mdata.obs_names.str.split("-").str[1].unique().tolist()
print(f"Found samples: {sample_nums}")

mdata.obs["sample"] = mdata.obs_names.str.split("-").str[1]

Found samples: ['6', '4', '5', '1', '3', '7']


In [8]:
multigrntools_raw_data_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS")

cell_type = "mouse_liver"

for sample_num in sample_nums:
    
    sample_mdata = mdata[mdata.obs["sample"] == sample_num].copy()
    
    sample_name = f"liver_{sample_num}"
    
    rna_csv_file = multigrntools_raw_data_dir / cell_type / sample_name / f"{sample_name}_RNA.csv"
    atac_csv_file = multigrntools_raw_data_dir / cell_type / sample_name / f"{sample_name}_ATAC.csv"
    
    rna_csv_file.parent.mkdir(parents=True, exist_ok=True)
    atac_csv_file.parent.mkdir(parents=True, exist_ok=True)

    print(f"Processing sample {sample_num}...")
    mu.pp.intersect_obs(sample_mdata)

    rna = sample_mdata.mod["rna"]
    rna_gene_by_cell_counts = pd.DataFrame(rna.layers["counts"].todense().T, index=rna.var_names, columns=rna.obs_names)

    atac = sample_mdata.mod["atac"]
    atac_peak_by_cell_counts = pd.DataFrame(atac.layers["counts"].todense().T, index=atac.var_names, columns=atac.obs_names)

    print(f"RNA cells: {rna.n_obs}, RNA genes: {rna.n_vars}")
    print(f"ATAC cells: {atac.n_obs}, ATAC peaks: {atac.n_vars}")

    print(f"Saving RNA data to {rna_csv_file}")
    rna_gene_by_cell_counts.to_csv(rna_csv_file, index=True)
    print(f"Saving ATAC data to {atac_csv_file}")
    atac_peak_by_cell_counts.to_csv(atac_csv_file, index=True)

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Processing sample 6...


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


RNA cells: 3067, RNA genes: 19265
ATAC cells: 3067, ATAC peaks: 149393
Saving RNA data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/liver/liver_6/liver_6_RNA.csv
Saving ATAC data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/liver/liver_6/liver_6_ATAC.csv


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


...

RNA cells: 2989, RNA genes: 19265
ATAC cells: 2989, ATAC peaks: 149393
Saving RNA data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/liver/liver_3/liver_3_RNA.csv
Saving ATAC data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/liver/liver_3/liver_3_ATAC.csv


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Processing sample 7...


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


RNA cells: 2762, RNA genes: 19265
ATAC cells: 2762, ATAC peaks: 149393
Saving RNA data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/liver/liver_7/liver_7_RNA.csv
Saving ATAC data to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/liver/liver_7/liver_7_ATAC.csv
